# Data Science Project - Phase 2: Data Cleaning
**The Context Gap**
Mohammad Qasim Khalaf Alsamarrae

In this notebook I clean the three datasets from Phase 1 and merge them into one table.

## Load the data

In [1]:
# download again in case the files are not in this session
!wget -nc -q https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip -O vqa.zip
!wget -nc -q https://vizwiz.cs.colorado.edu/VizWiz_final/image_quality/annotations.zip -O quality.zip
!wget -nc -q https://vizwiz.cs.colorado.edu/VizWiz_final/caption/annotations.zip -O captions.zip
!unzip -j -o -q vqa.zip -d vqa
!unzip -j -o -q quality.zip -d quality
!unzip -j -o -q captions.zip -d captions

In [2]:
# Import Pandas and numpy
import numpy as np
import pandas as pd

vqa = pd.read_json("vqa/val.json")
quality = pd.read_json("quality/val.json")
cap = pd.read_json("captions/val.json", typ="series")
cap_images = pd.DataFrame(cap["images"])
captions = pd.DataFrame(cap["annotations"])

print(vqa.shape, quality.shape, cap_images.shape, captions.shape)

(4319, 5) (7750, 3) (7750, 4) (38750, 6)


## 1. Cleaning the questions

In [3]:
# check empty values
vqa.isnull().sum()

,0
image,0
question,0
answers,0
answer_type,0
answerable,0


In [4]:
# check duplicated rows
vqa.duplicated(subset=["image", "question"]).sum()

np.int64(0)

In [5]:
# questions with extra spaces at the start, end or in the middle
extra_spaces = vqa["question"] != vqa["question"].str.split().str.join(" ")
extra_spaces.sum()

np.int64(339)

In [6]:
# clean version: remove extra spaces and make it lowercase
vqa["question_clean"] = vqa["question"].str.split().str.join(" ").str.lower()
vqa[["question", "question_clean"]].head()

,question,question_clean
0,Ok. There is another picture I hope it is a be...,ok. there is another picture i hope it is a be...
1,Can you tell me what this medicine is please?,can you tell me what this medicine is please?
2,What is the title of this book?,what is the title of this book?
3,Which one is the blue one?,which one is the blue one?
4,What does the arrow say?,what does the arrow say?


### Message type
Not every row is a clear question. I treat every row as a message and give it one class:
- **question**: a clear question ("What color is this?")
- **request**: no question mark, but a clear request or need ("Read this label", "Expiration date.")
- **other**: no clear need, like a follow-up after a failed try or random words ("testing, testing")

I don't delete "other" rows, because they show that the person had to try again.

In [7]:
start_words = ["what", "whats", "what's", "which", "who", "where", "when", "why", "how",
               "is", "are", "was", "does", "do", "did", "can", "could", "would", "will", "should", "may", "has", "have"]
request_words = ["read", "tell", "describe", "identify", "please", "help", "check", "name",
                 "need", "want", "know", "confirm", "see", "find", "looking", "show", "give",
                 "what", "whats", "what's", "which", "how",
                 "expiration", "date", "flavor", "color", "colour", "directions", "instructions",
                 "number", "code", "brand", "price", "kind", "type", "title", "label", "say", "says",
                 "ingredients", "temperature", "size"]

def message_type(q):
    words = q.replace("?", " ").replace(",", " ").replace(".", " ").split()
    if len(words) == 0:
        return "other"
    if "?" in q or words[0] in start_words:
        return "question"
    for w in words:
        if w in request_words:
            return "request"
    return "other"

vqa["message_type"] = vqa["question_clean"].apply(message_type)
vqa["message_type"].value_counts()

,count
message_type,
question,3978
request,266
other,75


In [8]:
# check some of each class
for t in ["request", "other"]:
    print(t)
    print(vqa[vqa["message_type"] == t]["question"].head(8).to_string())
    print()

request
25    I was wondering how ominous the sky is looking...
37    Hi, I just wanted to confirm that these are in...
38                         Please tell me what this is.
40    Please tell me what is in this food packaging....
43                        Tell me what's in this photo.
44    Able on my vacuum I need the serial number and...
60                 Please tell me the them of this book
63    Alright see if you can see the ORCA serial num...

other
0     Ok. There is another picture I hope it is a be...
24    Sometimes it does this too.  It's great when i...
50                                      Screen showing.
56                               Specials on this menu.
58    I cannot move the camera slightly closer to th...
66              Product key for this version of Office.
70                           About this one, thank you.
76                                       Business card.



## 2. Cleaning the answers
Each question has 10 answers. I clean them (lowercase, no punctuation, no "a / an / the"), then I count agreement in two ways:
- **agreement_exact**: how many volunteers wrote exactly the same answer. This is the official VQA rule: an answer is reliable if at least 3 people gave it.
- **agreement_meaning**: answers that describe the same thing also count, like "soup" and "tomato soup" (all the words of one answer are inside the other). "yes", "no" and "unanswerable" must match exactly.

When agreement_meaning is high but agreement_exact is low, the volunteers saw the same thing but gave different levels of detail.

The **top answer** is the answer written most often. If two answers are written the same number of times, I keep the longer one, because it gives more detail.

If the official label says the question **was answered** (answerable = 1), I ignore the "unanswerable" answers when I choose the top answer and count agreement. Otherwise "unanswerable" could win just because some volunteers could not see the photo well, while the others gave a real answer.

In [9]:
def clean_answer(text):
    text = text.lower().replace(".", " ").replace(",", " ").replace("!", " ").replace("'", "")
    words = [w for w in text.split() if w not in ["a", "an", "the"]]
    return " ".join(words)

def same_thing(a, b):
    if a == b:
        return True
    exact = ["yes", "no", "unanswerable", ""]
    if a in exact or b in exact:
        return False
    wa = set(a.split())
    wb = set(b.split())
    return wa <= wb or wb <= wa   # one answer is inside the other

def best_answer(answers, answerable):
    texts = [clean_answer(a["answer"]) for a in answers]
    # if the question was answered, use only the real answers
    if answerable == 1:
        real = [t for t in texts if t != "unanswerable"]
        if len(real) > 0:
            texts = real
    counts = pd.Series(texts).value_counts()
    # exact agreement: the answer written the most times
    agreement_exact = counts.max()
    # agreement by meaning: the biggest group of answers that describe the same thing
    agreement_meaning = max(sum(same_thing(t, other) for other in texts) for t in texts)
    # top answer: the answer written most often (longer one if there is a tie)
    top = counts[counts == counts.max()].index
    top_answer = max(top, key=len)
    return top_answer, agreement_exact, agreement_meaning

# apply runs the function on every row (axis=1 means row by row)
result = vqa.apply(lambda row: best_answer(row["answers"], row["answerable"]), axis=1)
vqa["top_answer"] = result.str[0]
vqa["agreement_exact"] = result.str[1]
vqa["agreement_meaning"] = result.str[2]
vqa[["question_clean", "top_answer", "agreement_exact", "agreement_meaning"]].head()

,question_clean,top_answer,agreement_exact,agreement_meaning
0,ok. there is another picture i hope it is a be...,unanswerable,8,8
1,can you tell me what this medicine is please?,night time,5,7
2,what is the title of this book?,dog years,9,9
3,which one is the blue one?,right,5,9
4,what does the arrow say?,unanswerable,8,8


### Checking that the data does not contradict itself
answerable and answer_type say almost the same thing in two ways. If answerable is 0, answer_type should be "unanswerable", and the other way around. I count the rows where they disagree.

In [10]:
conflict = (vqa["answerable"] == 0) != (vqa["answer_type"] == "unanswerable")
print("rows that contradict:", conflict.sum())
vqa[conflict][["question", "answerable", "answer_type", "top_answer"]]

rows that contradict: 0


,question,answerable,answer_type,top_answer


If there are any, I trust **answerable**, because it is the official label of the dataset.

In [11]:
# 10 = all volunteers agree, 1 = all different
print(vqa["agreement_exact"].value_counts().sort_index())
print(vqa["agreement_meaning"].value_counts().sort_index())

agreement_exact
1     292
2     474
3     478
4     389
5     487
6     479
7     474
8     482
9     446
10    318
Name: count, dtype: int64
agreement_meaning
1      81
2     212
3     253
4     289
5     461
6     550
7     545
8     614
9     680
10    634
Name: count, dtype: int64


In [12]:
# same thing seen, but different levels of detail
detail_varies = (vqa["agreement_exact"] <= 2) & (vqa["agreement_meaning"] >= 3)
detail_varies.sum()

np.int64(473)

In [13]:
# the answers list is not needed anymore
vqa = vqa.drop(columns=["answers"])
vqa.head()

,image,question,answer_type,answerable,question_clean,message_type,top_answer,agreement_exact,agreement_meaning
0,VizWiz_val_00000000.jpg,Ok. There is another picture I hope it is a be...,unanswerable,0,ok. there is another picture i hope it is a be...,other,unanswerable,8,8
1,VizWiz_val_00000001.jpg,Can you tell me what this medicine is please?,other,1,can you tell me what this medicine is please?,question,night time,5,7
2,VizWiz_val_00000002.jpg,What is the title of this book?,other,1,what is the title of this book?,question,dog years,9,9
3,VizWiz_val_00000003.jpg,Which one is the blue one?,other,1,which one is the blue one?,question,right,5,9
4,VizWiz_val_00000004.jpg,What does the arrow say?,unanswerable,0,what does the arrow say?,question,unanswerable,8,8


## 3. Cleaning the quality data

In [14]:
flaws = pd.json_normalize(quality["flaws"])
quality = pd.concat([quality.drop(columns="flaws"), flaws], axis=1)

print(quality.isnull().sum().sum())
print(quality.duplicated(subset=["image"]).sum())

0
0


In [15]:
# how often each problem appears (3 or more of 5 workers)
problems = ["BLR", "DRK", "BRT", "FRM", "OBS", "ROT", "OTH", "NON"]
((quality[problems] >= 3).mean() * 100).round(1)

,0
BLR,29.9
DRK,2.8
BRT,2.6
FRM,36.5
OBS,1.5
ROT,10.9
OTH,0.1
NON,29.4


**OTH** means "other problem". It almost never appears, and when it does we don't know what the problem was. A column that almost never changes does not help to find any relationship, so I remove it.

In [16]:
quality = quality.drop(columns=["OTH"])
quality.columns

Index(['image', 'unrecognizable', 'FRM', 'BLR', 'DRK', 'BRT', 'OBS', 'NON',
       'ROT'],
      dtype='object')

In [17]:
# keep only the photos that have questions
quality = quality[quality["image"].isin(vqa["image"])]
quality.shape

(4319, 9)

## 4. Cleaning the captions
Some captions can't be used:
- **is_rejected**: spam captions
- **is_precanned**: the worker wrote the ready sentence "Quality issues are too severe to recognize visual content."

Spam is useless, so I just remove it. But the ready sentence means a worker could not describe the photo, and that is useful information. So before removing the ready sentences, I count them for each photo.

In [18]:
# keep only the captions of my photos
cap_images = cap_images.rename(columns={"id": "image_id", "file_name": "image"})
captions = captions.merge(cap_images[["image_id", "image"]], on="image_id")
captions = captions[captions["image"].isin(vqa["image"])]
captions.shape

(21595, 7)

In [19]:
print("rejected:", captions["is_rejected"].sum())
print("precanned:", captions["is_precanned"].sum())
print("both:", (captions["is_rejected"] & captions["is_precanned"]).sum())

rejected: 531
precanned: 2571
both: 288


In [20]:
# remove spam first
captions = captions[~captions["is_rejected"]]

# count the ready sentences for each photo (0 to 5 workers)
precanned = captions.groupby("image")["is_precanned"].sum().reset_index(name="precanned_count")
precanned["precanned_count"].value_counts().sort_index()

,count
precanned_count,
0,3156
1,609
2,228
3,146
4,120
5,60


In [21]:
# now remove the ready sentences too
print("removed:", captions["is_precanned"].sum())
captions = captions[~captions["is_precanned"]]
captions.shape

removed: 2283


(18781, 7)

In [22]:
# clean the caption text
captions["caption"] = captions["caption"].str.replace("\r", "").str.split().str.join(" ")
captions["caption"].head()

,caption
0,A computer screen shows a repair prompt on the...
1,a computer screen with a repair automatically ...
2,partial computer screen showing the need of re...
3,Part of a computer monitor showing a computer ...
4,The top of a laptop with a blue background and...


In [23]:
# put the captions of each photo together in one cell and count them
# groupby groups the rows of the same photo
cap_text = captions.groupby("image")["caption"].apply(" | ".join).reset_index()
cap_count = captions.groupby("image").size().reset_index(name="valid_captions")
cap_text.head()

,image,caption
0,VizWiz_val_00000000.jpg,A computer screen shows a repair prompt on the...
1,VizWiz_val_00000001.jpg,A person is holding a bottle that has medicine...
2,VizWiz_val_00000002.jpg,a white paper showing an image of black and br...
3,VizWiz_val_00000003.jpg,A white box is to the left of a blue box on a ...
4,VizWiz_val_00000004.jpg,close up of a computer monitor that is powered...


## 5. Merging everything into one table

In [24]:
# merge joins two tables using the image name
data = vqa.merge(quality, on="image", how="left")
data = data.merge(cap_images[["image", "text_detected"]], on="image", how="left")
data = data.merge(cap_text, on="image", how="left")
data = data.merge(cap_count, on="image", how="left")
data = data.merge(precanned, on="image", how="left")
data.shape

(4319, 21)

In [25]:
data.isnull().sum()

,0
image,0
question,0
answer_type,0
answerable,0
question_clean,0
message_type,0
top_answer,0
agreement_exact,0
agreement_meaning,0
unrecognizable,0


In [26]:
# photos with no valid caption left: i keep them, because a photo nobody
# could describe is important for my project
data["caption"] = data["caption"].fillna("")
data["valid_captions"] = data["valid_captions"].fillna(0).astype(int)
data["precanned_count"] = data["precanned_count"].fillna(0).astype(int)
data["valid_captions"].value_counts().sort_index()

,count
valid_captions,
0,75
1,124
2,155
3,312
4,854
5,2799


In [27]:
data.isnull().sum().sum()

np.int64(0)

Rule: a column that is more than half empty does not help, so I remove it. Here I check every column.

In [28]:
empty_percent = data.isnull().mean() * 100
too_empty = empty_percent[empty_percent > 50].index
print("columns removed:", list(too_empty))
data = data.drop(columns=too_empty)

columns removed: []


In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4319 entries, 0 to 4318
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   image              4319 non-null   object
 1   question           4319 non-null   object
 2   answer_type        4319 non-null   object
 3   answerable         4319 non-null   int64 
 4   question_clean     4319 non-null   object
 5   message_type       4319 non-null   object
 6   top_answer         4319 non-null   object
 7   agreement_exact    4319 non-null   int64 
 8   agreement_meaning  4319 non-null   int64 
 9   unrecognizable     4319 non-null   int64 
 10  FRM                4319 non-null   int64 
 11  BLR                4319 non-null   int64 
 12  DRK                4319 non-null   int64 
 13  BRT                4319 non-null   int64 
 14  OBS                4319 non-null   int64 
 15  NON                4319 non-null   int64 
 16  ROT                4319 non-null   int64 


In [30]:
data.head()

,image,question,answer_type,answerable,question_clean,message_type,top_answer,agreement_exact,agreement_meaning,unrecognizable,...,BLR,DRK,BRT,OBS,NON,ROT,text_detected,caption,valid_captions,precanned_count
0,VizWiz_val_00000000.jpg,Ok. There is another picture I hope it is a be...,unanswerable,0,ok. there is another picture i hope it is a be...,other,unanswerable,8,8,0,...,1,0,0,0,0,0,True,A computer screen shows a repair prompt on the...,5,0
1,VizWiz_val_00000001.jpg,Can you tell me what this medicine is please?,other,1,can you tell me what this medicine is please?,question,night time,5,7,0,...,5,0,0,0,0,0,True,A person is holding a bottle that has medicine...,5,0
2,VizWiz_val_00000002.jpg,What is the title of this book?,other,1,what is the title of this book?,question,dog years,9,9,0,...,0,0,0,0,2,0,True,a white paper showing an image of black and br...,5,0
3,VizWiz_val_00000003.jpg,Which one is the blue one?,other,1,which one is the blue one?,question,right,5,9,0,...,3,0,0,0,2,0,True,A white box is to the left of a blue box on a ...,5,0
4,VizWiz_val_00000004.jpg,What does the arrow say?,unanswerable,0,what does the arrow say?,question,unanswerable,8,8,0,...,0,0,0,1,3,1,True,close up of a computer monitor that is powered...,5,0


## Save

In [31]:
data.to_csv("clean_data.csv", index=False)

## Notes
- No duplicates and no missing values in the questions and answers, and answerable / answer_type never contradict each other (0 rows).
- 339 questions had extra spaces. I made a clean lowercase version and kept the original.
- Message type: 3,978 questions, 266 requests, 75 other (no clear need). I kept the "other" rows because they show the person had to try again. A few of them are short requests like "Business card.", so the rule is not perfect.
- I count agreement in two ways. agreement_exact follows the official VQA rule (exact same answer after cleaning): 766 questions (17.7%) have 2 or fewer volunteers agreeing, and in 318 all 10 agree. agreement_meaning also counts answers like "soup" and "tomato soup": 293 questions (6.8%) have 2 or fewer agreeing.
- In 473 questions the exact agreement is low (2 or fewer) but the meaning agreement is 3 or more: the volunteers saw the same thing but gave different levels of detail.
- The top answer is the one written most often (the longer one if there is a tie). When the official label says the question was answered, I ignore the "unanswerable" answers. Without this, "unanswerable" was the top answer in 521 answered questions, often only because it is a long word and won the tie.
- I did not use the volunteers' confidence, because it will not exist when the AI layer is used in real life, and agreement already shows the same idea.
- I removed the OTH column ("other problem"). It is 0 in 91.9% of photos and reached 3 or more workers in only 9 of 7,750 photos.
- No column was more than half empty, so no other column was removed.
- I removed 531 spam captions. 2,571 were the ready sentence "quality too severe" (288 were also spam). Before removing them I counted them per photo: 1,163 photos (26.9%) have at least one, and 60 photos have all five.
- 18,781 captions are left. 75 photos have no valid caption, and I kept them with 0 captions.
- The final clean table has 4,319 rows and 21 columns, with no missing values.

Next step: creating my own features and the target (the context gap).